In [0]:
from pyspark.sql import functions as sf
from pyspark.sql.types import DoubleType

df = spark.table("medical_data.bronze.encounters") \
    .select("id", "start", "stop", "patient", "organization", "payer", "encounter_class", "code", "description", "base_encounter_cost", "total_claim_cost", "payer_coverage")

# Clean numeric columns and fill nulls
df_clean = df \
    .withColumn("base_encounter_cost", sf.regexp_replace(sf.col("base_encounter_cost"), ",", "").cast(DoubleType())) \
    .withColumn("total_claim_cost", sf.regexp_replace(sf.col("total_claim_cost"), ",", "").cast(DoubleType())) \
    .withColumn("payer_coverage", sf.regexp_replace(sf.col("payer_coverage"), ",", "").cast(DoubleType())) \
    .fillna({
        "base_encounter_cost": 0,
        "total_claim_cost": 0,
        "payer_coverage": 0,
        "organization": "NA",
        "payer": "NA",
        "encounter_class": "NA",
        "code": "NA",
        "description": "NA"
    })

# Remove duplicates and filter null ids
df_clean = df_clean.dropDuplicates(["id"]).filter(sf.col("id").isNotNull())

# Feature engineering
df_with_features = df_clean \
    .withColumn("month", sf.month(sf.to_timestamp("stop"))) \
    .withColumn("quarter", sf.quarter(sf.to_timestamp("stop"))) \
    .withColumn("time_diff", (sf.unix_timestamp("stop") - sf.unix_timestamp("start"))/3600)

df_with_features.write.mode("overwrite").saveAsTable("medical_data.silver.encounters_s")